In [1]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 66.0 MB/s eta 0:00:00


In [2]:
!pip install datasets[vision]

In [7]:
from google.colab import userdata
token = userdata.get('HF_SECRET')

In [8]:
from datasets import load_dataset
import json
from pycocotools import mask as mask_utils

ds = load_dataset("sebnae/IndoorCrowd", "obj_det_seg", split="acs_ec", token=token)
row = ds[0]
rle = json.loads(row["objects"]["rle_mask"][0])
mask = mask_utils.decode(rle)

obj_det_seg/acs_ec-00000-of-00002.parque(…): reconstructing file:   0%|          |  0.00B /  331MB            

obj_det_seg/acs_ec-00000-of-00002.parque(…): downloading bytes:           |  0.00B            

obj_det_seg/acs_ec-00001-of-00002.parque(…): reconstructing file:   0%|          |  0.00B /  347MB            

obj_det_seg/acs_ec-00001-of-00002.parque(…): downloading bytes:           |  0.00B            

obj_det_seg/acs_eg-00000-of-00002.parque(…): reconstructing file:   0%|          |  0.00B /  321MB            

...

obj_det_seg/r_central-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating acs_ec split:   0%|          | 0/2802 [00:00<?, ? examples/s]

Generating acs_eg split:   0%|          | 0/3975 [00:00<?, ? examples/s]

Generating ie_central split:   0%|          | 0/1650 [00:00<?, ? examples/s]

Generating r_central split:   0%|          | 0/380 [00:00<?, ? examples/s]

# Prepare the directory structure for YOLO dataset.

In [10]:
from pathlib import Path
import shutil

# Define the base directory for your YOLO dataset
YOLO_DATA_DIR = Path("yolo_dataset")

# Create train, val, and test directories for images and labels
(YOLO_DATA_DIR / "images" / "train").mkdir(parents=True, exist_ok=True)
(YOLO_DATA_DIR / "labels" / "train").mkdir(parents=True, exist_ok=True)
(YOLO_DATA_DIR / "images" / "val").mkdir(parents=True, exist_ok=True)
(YOLO_DATA_DIR / "labels" / "val").mkdir(parents=True, exist_ok=True)
(YOLO_DATA_DIR / "images" / "test").mkdir(parents=True, exist_ok=True)
(YOLO_DATA_DIR / "labels" / "test").mkdir(parents=True, exist_ok=True)

print(f"Created directory structure at {YOLO_DATA_DIR}")

Created directory structure at yolo_dataset


# Iterate through each image, decode the RLE masks, convert them to polygons, and save the images and their corresponding YOLO format label files.

In [11]:
import numpy as np
from PIL import Image, ImageDraw
import cv2
from sklearn.model_selection import train_test_split
from tqdm import tqdm # Import tqdm

def rle_to_polygon(rle_mask):
    mask = mask_utils.decode(rle_mask)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)

    polygons = []
    for contour in contours:
        # Reshape to a 1D array of (x1, y1, x2, y2, ...) coordinates
        polygon = contour.flatten().tolist()

        # A polygon must have at least 3 points (6 coordinates).
        if len(polygon) >= 6:
            polygons.append(polygon)
    return polygons


def save_yolo_labels(image_id, polygons, image_width, image_height, output_dir):
    with open(output_dir / f"{image_id}.txt", "w") as f:
        for polygon in polygons:
            # Normalize coordinates
            normalized_polygon = [str(coord / (image_width if i % 2 == 0 else image_height)) for i, coord in enumerate(polygon)]
            # YOLO format: <class_id> <x1> <y1> <x2> <y2> ...
            # Assuming a single class for simplicity (class_id 0)
            f.write(f"0 {' '.join(normalized_polygon)}\n")

def process_and_save_split(dataset_split, split_name, yolo_data_dir):
    print(f"Processing {split_name} split...")
    # Wrap the loop with tqdm for a progress bar
    for i, row in enumerate(tqdm(dataset_split, desc=f"Saving {split_name} images and labels")):
        image = row["image"]
        image_id = row["image_id"]
        width, height = row["width"], row["height"]

        # Save image
        image.save(yolo_data_dir / "images" / split_name / f"{image_id}.jpg")

        # Process masks
        all_polygons = []
        for rle_mask_data in row["objects"]["rle_mask"]:
            rle = json.loads(rle_mask_data)
            polygons = rle_to_polygon(rle)
            all_polygons.extend(polygons)

        if all_polygons:
            save_yolo_labels(image_id, all_polygons, width, height, yolo_data_dir / "labels" / split_name)
    print(f"Finished processing {split_name} split.")

# Split the dataset into train, validation, and test sets
# Using an 80/10/10 split for train/val/test

train_val_test_split = ds.train_test_split(test_size=0.2, seed=42) # 80% train, 20% for val+test
val_test_split = train_val_test_split['test'].train_test_split(test_size=0.5, seed=42) # 10% val, 10% test

train_ds = train_val_test_split['train']
val_ds = val_test_split['train']
test_ds = val_test_split['test']

print(f"Train set size: {len(train_ds)}")
print(f"Validation set size: {len(val_ds)}")
print(f"Test set size: {len(test_ds)}")

# Process and save each split
process_and_save_split(train_ds, "train", YOLO_DATA_DIR)
process_and_save_split(val_ds, "val", YOLO_DATA_DIR)
process_and_save_split(test_ds, "test", YOLO_DATA_DIR)

print("Dataset processing complete.")

Train set size: 2241
Validation set size: 280
Test set size: 281
Processing train split...


Saving train images and labels: 100%|██████████| 2241/2241 [03:22<00:00, 11.06it/s]


Finished processing train split.
Processing val split...


Saving val images and labels: 100%|██████████| 280/280 [00:24<00:00, 11.48it/s]


Finished processing val split.
Processing test split...


Saving test images and labels: 100%|██████████| 281/281 [00:24<00:00, 11.32it/s]

Finished processing test split.
Dataset processing complete.


# Create a `data.yaml` file that tells YOLO where to find the dataset and what classes are present.

In [12]:
data_yaml_content = f'''
path: {YOLO_DATA_DIR.resolve()}
train: images/train
val: images/val
test: images/test

# Classes
nc: 1  # Number of classes
names: ['object']  # Class names
'''

with open(YOLO_DATA_DIR / "data.yaml", "w") as f:
    f.write(data_yaml_content)

print(f"Created data.yaml at {YOLO_DATA_DIR / 'data.yaml'}")

Created data.yaml at yolo_dataset/data.yaml


# Upload the dataset to hunggingface (Optional)

In [13]:
# from huggingface_hub import login, upload_folder

# (optional) Login with your Hugging Face credentials
# login()

# Push your dataset files
# upload_folder(folder_path=YOLO_DATA_DIR, repo_id="jakekato/IndoorCrowdYoloFormat", repo_type="dataset")

# Actually read and finetune the model

In [14]:
from ultralytics import YOLO
from pathlib import Path

# MODEL_NAME = "yolo26n"
MODEL_NAME = "yolov8n"
OUTPUT_NAME = f"{MODEL_NAME}-finetuned"

model = YOLO(f"{MODEL_NAME}-seg.pt")

results = model.train(
    data=YOLO_DATA_DIR / "data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    name=OUTPUT_NAME,
)

print(results.save_dir)

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.118 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=

In [15]:
from huggingface_hub import login, upload_folder
login(token=token)
upload_folder(folder_path=results.save_dir, repo_id="jakekato/IndoorCrowdYoloFormat", repo_type="dataset")

CommitInfo(commit_url='https://huggingface.co/datasets/jakekato/IndoorCrowdYoloFormat/commit/5153d55999208f99a0b93db1a0c1829027b81fb5', commit_message='Upload folder using huggingface_hub', commit_description='', oid='5153d55999208f99a0b93db1a0c1829027b81fb5', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/jakekato/IndoorCrowdYoloFormat', endpoint='https://huggingface.co', repo_type='dataset', repo_id='jakekato/IndoorCrowdYoloFormat'), pr_revision=None, pr_num=None)